## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV)
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score, f1_score)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBRegressor, XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

from skopt import BayesSearchCV
from skopt.space import Integer, Categorical, Real

import warnings
warnings.filterwarnings('ignore')

## Importing Dataset

In [ ]:
#Load regression datasets
X_train_reg_scaled = pd.read_pickle('X_train_reg_scaled.pkl')
X_val_reg_scaled = pd.read_pickle('X_val_reg_scaled.pkl')
X_test_reg_scaled = pd.read_pickle('X_test_reg_scaled.pkl')
y_train_reg = pd.read_pickle('y_train_reg.pkl')
y_val_reg = pd.read_pickle('y_val_reg.pkl')
y_test_reg = pd.read_pickle('y_test_reg.pkl')

In [ ]:
#Load classification datasets
X_train_clf_scaled = pd.read_pickle('X_train_clf_scaled.pkl')
X_val_clf_scaled = pd.read_pickle('X_val_clf_scaled.pkl')
X_test_clf_scaled = pd.read_pickle('X_test_clf_scaled.pkl')
y_train_clf = pd.read_pickle('y_train_clf.pkl')
y_val_clf = pd.read_pickle('y_val_clf.pkl')
y_test_clf = pd.read_pickle('y_test_clf.pkl')

In [ ]:
#Verify shapes
print("Regression split shapes:", X_train_reg_scaled.shape, X_val_reg_scaled.shape, X_test_reg_scaled.shape)
print("Classification split shapes:", X_train_clf_scaled.shape, X_val_clf_scaled.shape, X_test_clf_scaled.shape)

Regression split shapes: (36908, 82) (7909, 82) (7910, 82)
Classification split shapes: (36908, 82) (7909, 82) (7910, 82)


In [ ]:
print("Regression target shapes:", y_train_reg.shape, y_val_reg.shape, y_test_reg.shape)
print("Classification target shapes:", y_train_clf.shape, y_val_clf.shape, y_test_clf.shape)

Regression target shapes: (36908,) (7909,) (7910,)
Classification target shapes: (36908,) (7909,) (7910,)


## Regression Models with Hyperparameter Tuning

In [ ]:
def print_reg_metrics(model_name, y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred):

    #Compute RMSE for train, validation, and test sets
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    #Compute R^2 scores for train, validation, and test sets
    train_r2 = r2_score(y_train, y_train_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    #Print results in a readable format
    print(f"{model_name} Regression:")
    print(f"Train RMSE: {train_rmse:.2f}, Val RMSE: {val_rmse:.2f}, Test RMSE: {test_rmse:.2f}")
    print(f"Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}, Test R2: {test_r2:.4f}\n")

    #Return results in a dictionary for later use
    return {
        'Model': model_name,
        'Train RMSE': train_rmse,
        'Val RMSE': val_rmse,
        'Test RMSE': test_rmse,
        'Train R2': train_r2,
        'Val R2': val_r2,
        'Test R2': test_r2
    }

#### Linear Regression

In [ ]:
#Initialize the Linear Regression model
lr = LinearRegression()

#Fit the model on the scaled training data
lr.fit(X_train_reg_scaled, y_train_reg)

#Predict on the scaled training set
lr_train_pred = lr.predict(X_train_reg_scaled)

#Predict on the scaled validation set
lr_val_pred = lr.predict(X_val_reg_scaled)

#Predict on the scaled test set
lr_test_pred = lr.predict(X_test_reg_scaled)

#Evaluate and store the metrics using the custom function
lr_results = print_reg_metrics(
    "Linear Regression",
    y_train_reg, lr_train_pred,
    y_val_reg, lr_val_pred,
    y_test_reg, lr_test_pred
)

Linear Regression Regression:
Train RMSE: 5466362.65, Val RMSE: 4757050.54, Test RMSE: 5742351.21
Train R2: 0.6035, Val R2: 0.7345, Test R2: 0.5355



#### Random Forest - Grid Search

In [ ]:
#Define the hyperparameter grid for Random Forest
rf_params = {
    'n_estimators': [100, 200],        #Number of trees in the forest
    'max_depth': [10, 20],             #Maximum depth of each tree
    'min_samples_split': [2, 5]        #Minimum number of samples required to split an internal node
}

#Set up GridSearchCV with 5-fold cross-validation and R² as the scoring metric
grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=5,
    scoring='r2',
    n_jobs=-1  # Use all available cores
)

#Fit the grid search on the scaled training data
grid_rf.fit(X_train_reg_scaled, y_train_reg)

#Retrieve the best model from grid search
rf_best = grid_rf.best_estimator_

#Predict using the best model on all datasets
rf_train_pred = rf_best.predict(X_train_reg_scaled)
rf_val_pred = rf_best.predict(X_val_reg_scaled)
rf_test_pred = rf_best.predict(X_test_reg_scaled)

#Evaluate and store the results using the custom metrics function
rf_results = print_reg_metrics(
    "Random Forest (Grid Search)",
    y_train_reg, rf_train_pred,
    y_val_reg, rf_val_pred,
    y_test_reg, rf_test_pred
)

#Print the best hyperparameters found by grid search
print("Best RF Params:", grid_rf.best_params_)

Random Forest (Grid Search) Regression:
Train RMSE: 330531.96, Val RMSE: 919789.53, Test RMSE: 650026.34
Train R2: 0.9986, Val R2: 0.9901, Test R2: 0.9940

Best RF Params: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}


#### XGBoost - Random Search

In [ ]:
#Define the parameter distribution to sample from
xgb_params = {
    'n_estimators': [100, 200],         #Number of boosting rounds
    'max_depth': [3, 6, 10],            #Depth of each tree
    'learning_rate': [0.01, 0.1, 0.3]   #Learning rate (eta)
}

#Initialize RandomizedSearchCV with 10 random combinations
rand_xgb = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),   #Base model
    param_distributions=xgb_params,            #Hyperparameter distribution
    n_iter=10,                                 #Number of random combinations to try
    cv=5,                                      #5-fold cross-validation
    scoring='r2',                              #Evaluation metric: R²
    n_jobs=-1,                                 #Use all available CPU cores
    random_state=42                            #Reproducibility
)

#Fit the randomized search on the scaled training data
rand_xgb.fit(X_train_reg_scaled, y_train_reg)

#Get the best model found during the search
xgb_best = rand_xgb.best_estimator_

#Make predictions using the best XGBoost model
xgb_train_pred = xgb_best.predict(X_train_reg_scaled)
xgb_val_pred = xgb_best.predict(X_val_reg_scaled)
xgb_test_pred = xgb_best.predict(X_test_reg_scaled)

#Evaluate and store metrics
xgb_results = print_reg_metrics(
    "XGBoost (Random Search)",
    y_train_reg, xgb_train_pred,
    y_val_reg, xgb_val_pred,
    y_test_reg, xgb_test_pred
)

#Output the best parameters found
print("Best XGB Params:", rand_xgb.best_params_)

XGBoost (Random Search) Regression:
Train RMSE: 150218.16, Val RMSE: 338142.23, Test RMSE: 313629.55
Train R2: 0.9997, Val R2: 0.9987, Test R2: 0.9986

Best XGB Params: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.3}


#### Neural Network - Grid Search

In [ ]:
#Scale the target variable (y values)
scaler_y = StandardScaler()
y_train_reg_scaled = scaler_y.fit_transform(y_train_reg.values.reshape(-1, 1)).flatten()
y_val_reg_scaled = scaler_y.transform(y_val_reg.values.reshape(-1, 1)).flatten()
y_test_reg_scaled = scaler_y.transform(y_test_reg.values.reshape(-1, 1)).flatten()

#Define a function to build a neural network with variable layers and learning rate
def build_nn(hidden_layers=(100, 50), learning_rate=0.001):
    model = Sequential()
    model.add(Dense(hidden_layers[0], activation='relu', input_shape=(X_train_reg_scaled.shape[1],)))
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation='relu'))
    model.add(Dense(1))  # Output layer for regression
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    return model

#Define the hyperparameter grid
nn_params = {
    'hidden_layers': [(100, 50), (200, 100, 50), (100, 100)],  #Network architectures
    'learning_rate': [0.001, 0.01],                            #Learning rates to try
    'batch_size': [32, 64]                                     #Batch sizes to try
}

#Initialize tracking variables
best_nn = None
best_val_rmse = float('inf')
best_params = None

#Manual grid search with early stopping
for layers in nn_params['hidden_layers']:
    for lr in nn_params['learning_rate']:
        for batch_size in nn_params['batch_size']:
            nn = build_nn(hidden_layers=layers, learning_rate=lr)
            early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            nn.fit(
                X_train_reg_scaled, y_train_reg_scaled,
                validation_data=(X_val_reg_scaled, y_val_reg_scaled),
                epochs=100,
                batch_size=batch_size,
                callbacks=[early_stopping],
                verbose=0
            )
            #Predict on validation set (scaled)
            val_pred_scaled = nn.predict(X_val_reg_scaled, verbose=0).flatten()
            val_pred = scaler_y.inverse_transform(val_pred_scaled.reshape(-1, 1)).flatten()
            val_rmse = np.sqrt(mean_squared_error(y_val_reg, val_pred))

            #Track the best model based on validation RMSE
            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_nn = nn
                best_params = {
                    'hidden_layers': layers,
                    'learning_rate': lr,
                    'batch_size': batch_size
                }

#Predict on all sets using the best model
nn_train_pred_scaled = best_nn.predict(X_train_reg_scaled, verbose=0).flatten()
nn_val_pred_scaled = best_nn.predict(X_val_reg_scaled, verbose=0).flatten()
nn_test_pred_scaled = best_nn.predict(X_test_reg_scaled, verbose=0).flatten()

#Inverse-transform predictions back to original scale
nn_train_pred = scaler_y.inverse_transform(nn_train_pred_scaled.reshape(-1, 1)).flatten()
nn_val_pred = scaler_y.inverse_transform(nn_val_pred_scaled.reshape(-1, 1)).flatten()
nn_test_pred = scaler_y.inverse_transform(nn_test_pred_scaled.reshape(-1, 1)).flatten()

#Evaluate and store metrics
nn_results = print_reg_metrics(
    "Neural Network (Grid Search)",
    y_train_reg, nn_train_pred,
    y_val_reg, nn_val_pred,
    y_test_reg, nn_test_pred
)

#Output best parameters found
print("Best NN Params:", best_params)

Neural Network (Grid Search) Regression:
Train RMSE: 621221.78, Val RMSE: 573281.26, Test RMSE: 615925.42
Train R2: 0.9949, Val R2: 0.9961, Test R2: 0.9947

Best NN Params: {'hidden_layers': (100, 50), 'learning_rate': 0.01, 'batch_size': 64}


#### KNN - Bayesian Optimization

In [ ]:
#Define the search space for KNN hyperparameters
knn_params = {
    'n_neighbors': Integer(3, 15),                   #Try values from 3 to 15 for number of neighbors
    'weights': Categorical(['uniform', 'distance']), #Use either uniform or distance-based weights
    'p': Integer(1, 2)                               #1 = Manhattan distance, 2 = Euclidean distance
}

#Initialize BayesSearchCV with KNN and 5-fold cross-validation
bayes_knn = BayesSearchCV(
    estimator=KNeighborsRegressor(),  #Base KNN model
    search_spaces=knn_params,         #Hyperparameter search space
    n_iter=10,                        #Number of search iterations
    cv=5,                             #5-fold cross-validation
    scoring='r2',                     #Evaluation metric: R²
    n_jobs=-1,                        #Use all available CPU cores
    random_state=42                   #Reproducibility
)

#Fit the Bayesian search on the scaled training data
bayes_knn.fit(X_train_reg_scaled, y_train_reg)

#Get the best estimator from the search
knn_best = bayes_knn.best_estimator_

#Predict using the best KNN model
knn_train_pred = knn_best.predict(X_train_reg_scaled)
knn_val_pred = knn_best.predict(X_val_reg_scaled)
knn_test_pred = knn_best.predict(X_test_reg_scaled)

#Evaluate and store metrics
knn_results = print_reg_metrics(
    "KNN (Bayesian Opt)",
    y_train_reg, knn_train_pred,
    y_val_reg, knn_val_pred,
    y_test_reg, knn_test_pred
)

#Print the best parameters found during the search
print("Best KNN Params:", bayes_knn.best_params_)

KNN (Bayesian Opt) Regression:
Train RMSE: 0.00, Val RMSE: 399374.36, Test RMSE: 621618.57
Train R2: 1.0000, Val R2: 0.9981, Test R2: 0.9946

Best KNN Params: OrderedDict({'n_neighbors': 13, 'p': 1, 'weights': 'distance'})


## Regression Results

In [ ]:
#Combine all regression results into a list for easy comparison or tabulation
reg_results = [lr_results, rf_results, xgb_results, nn_results, knn_results]

## Classification Models with Hyperparameter Tuning

In [ ]:
def print_clf_metrics(model_name, y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred):
    #Compute accuracy for train, validation, and test sets
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    #Compute F1 score for train, validation, and test sets
    train_f1 = f1_score(y_train, y_train_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    #Print results in a readable format
    print(f"{model_name} Classification:")
    print(f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")
    print(f"Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}, Test F1: {test_f1:.4f}\n")

    #Return results in a dictionary for later use
    return {
        'Model': model_name,
        'Train Acc': train_acc,
        'Val Acc': val_acc,
        'Test Acc': test_acc,
        'Train F1': train_f1,
        'Val F1': val_f1,
        'Test F1': test_f1
    }

#### Logistic Regression - Grid Search

In [ ]:
#Define the grid of hyperparameters to search
lr_clf_params = {
    'C': [0.1, 1, 10],                       #Regularization strength (smaller = stronger regularization)
    'solver': ['lbfgs', 'liblinear']         #Optimization algorithms supported by Logistic Regression
}

#Set up GridSearchCV for Logistic Regression
grid_lr_clf = GridSearchCV(
    estimator=LogisticRegression(random_state=42),   #Base model
    param_grid=lr_clf_params,                        #Hyperparameter grid
    cv=5,                                            #5-fold cross-validation
    scoring='f1',                                    #Use F1 score for evaluation
    n_jobs=-1                                        #Use all CPU cores
)

#Fit the grid search on scaled training data
grid_lr_clf.fit(X_train_clf_scaled, y_train_clf)

#Extract the best model found by grid search
lr_clf_best = grid_lr_clf.best_estimator_

#Predict on train, validation, and test sets using the best model
lr_clf_train_pred = lr_clf_best.predict(X_train_clf_scaled)
lr_clf_val_pred = lr_clf_best.predict(X_val_clf_scaled)
lr_clf_test_pred = lr_clf_best.predict(X_test_clf_scaled)

#Evaluate classification metrics and store results
lr_clf_results = print_clf_metrics(
    "Logistic Regression (Grid Search)",
    y_train_clf, lr_clf_train_pred,
    y_val_clf, lr_clf_val_pred,
    y_test_clf, lr_clf_test_pred
)

#Output the best hyperparameters found
print("Best LR Params:", grid_lr_clf.best_params_)

/Users/fuadbh/Projects/CSE445/ML Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/fuadbh/Projects/CSE445/ML Project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.

Logistic Regression (Grid Search) Classification:
Train Acc: 0.8189, Val Acc: 0.8226, Test Acc: 0.8238
Train F1: 0.8740, Val F1: 0.8763, Test F1: 0.8780

Best LR Params: {'C': 1, 'solver': 'lbfgs'}


#### Random Forest - Random Search

In [ ]:
#Define hyperparameter grid for Random Forest Classifier
rf_clf_params = {
    'n_estimators': [100, 200],        #Number of trees in the forest
    'max_depth': [10, 20],             #Maximum depth of each tree
    'min_samples_split': [2, 5]        #Minimum samples required to split an internal node
}

#Perform Randomized Search with 5-fold cross-validation
rand_rf_clf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),  #Base estimator
    rf_clf_params,                            #Parameter grid
    n_iter=10,                                #Number of random combinations to try
    cv=5,                                     #5-fold cross-validation
    scoring='f1',                             #Optimize for F1 score
    n_jobs=-1,                                #Use all CPU cores
    random_state=42                           #Reproducibility
)

#Fit the model on the training data
rand_rf_clf.fit(X_train_clf_scaled, y_train_clf)

#Get the best model found during search
rf_clf_best = rand_rf_clf.best_estimator_

#Make predictions on all data splits
rf_clf_train_pred = rf_clf_best.predict(X_train_clf_scaled)
rf_clf_val_pred = rf_clf_best.predict(X_val_clf_scaled)
rf_clf_test_pred = rf_clf_best.predict(X_test_clf_scaled)

#Evaluate the performance using your custom classification metrics function
rf_clf_results = print_clf_metrics(
    "Random Forest (Random Search)",
    y_train_clf, rf_clf_train_pred,
    y_val_clf, rf_clf_val_pred,
    y_test_clf, rf_clf_test_pred
)

#Print the best hyperparameter combination found
print("Best RF Params:", rand_rf_clf.best_params_)

Random Forest (Random Search) Classification:
Train Acc: 1.0000, Val Acc: 0.9996, Test Acc: 0.9994
Train F1: 1.0000, Val F1: 0.9997, Test F1: 0.9995

Best RF Params: {'n_estimators': 100, 'min_samples_split': 2, 'max_depth': 20}


#### XGBoost - Bayesian Optimization

In [ ]:
#Define the hyperparameter search space
xgb_clf_params = {
    'n_estimators': Integer(100, 200),                    #Number of boosting rounds
    'max_depth': Integer(3, 10),                          #Maximum tree depth
    'learning_rate': Real(0.01, 0.3, prior='log-uniform') #Step size shrinkage
}

#Set up Bayesian Optimization search over the hyperparameters
bayes_xgb_clf = BayesSearchCV(
    XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),  # Base XGB classifier
    xgb_clf_params,           # Hyperparameter space
    n_iter=10,                # Number of combinations to try
    cv=5,                     # 5-fold cross-validation
    scoring='f1',             # Evaluation metric
    n_jobs=-1,                # Use all CPU cores
    random_state=42           # Reproducibility
)

#Fit model to training data
bayes_xgb_clf.fit(X_train_clf_scaled, y_train_clf)

#Extract the best model found during search
xgb_clf_best = bayes_xgb_clf.best_estimator_

#Predict on training, validation, and test data
xgb_clf_train_pred = xgb_clf_best.predict(X_train_clf_scaled)
xgb_clf_val_pred = xgb_clf_best.predict(X_val_clf_scaled)
xgb_clf_test_pred = xgb_clf_best.predict(X_test_clf_scaled)

#Evaluate model performance
xgb_clf_results = print_clf_metrics(
    "XGBoost (Bayesian Opt)",
    y_train_clf, xgb_clf_train_pred,
    y_val_clf, xgb_clf_val_pred,
    y_test_clf, xgb_clf_test_pred
)

#Output the best found hyperparameters
print("Best XGB Params:", bayes_xgb_clf.best_params_)

/Users/fuadbh/Projects/CSE445/ML Project/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [18:43:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/fuadbh/Projects/CSE445/ML Project/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [18:43:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/fuadbh/Projects/CSE445/ML Project/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [18:43:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/fuadbh/Projects/CSE445/ML Project/.venv/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [18:43:23] WARNING: /Users/runn

XGBoost (Bayesian Opt) Classification:
Train Acc: 1.0000, Val Acc: 0.9991, Test Acc: 0.9991
Train F1: 1.0000, Val F1: 0.9994, Test F1: 0.9994

Best XGB Params: OrderedDict({'learning_rate': 0.15171809566296443, 'max_depth': 6, 'n_estimators': 153})


#### Neural Network - Basic

In [ ]:
#Initialize the MLPClassifier with two hidden layers: one with 100 neurons, one with 50
#Set max_iter=500 to give it more training time
nn_clf = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    random_state=42
)

#Train the model on the scaled training set
nn_clf.fit(X_train_clf_scaled, y_train_clf)

#Make predictions on training, validation, and test sets
nn_clf_train_pred = nn_clf.predict(X_train_clf_scaled)
nn_clf_val_pred = nn_clf.predict(X_val_clf_scaled)
nn_clf_test_pred = nn_clf.predict(X_test_clf_scaled)

#Evaluate performance using a custom metric printing function
nn_clf_results = print_clf_metrics(
    "Neural Network",
    y_train_clf, nn_clf_train_pred,
    y_val_clf, nn_clf_val_pred,
    y_test_clf, nn_clf_test_pred
)

Neural Network Classification:
Train Acc: 0.8007, Val Acc: 0.7995, Test Acc: 0.7994
Train F1: 0.8403, Val F1: 0.8386, Test F1: 0.8401



#### KNN - Grid Search

In [ ]:
#Define the hyperparameter grid for KNN
#- n_neighbors: number of nearest neighbors to consider
#- weights: 'uniform' treats all neighbors equally, 'distance' weights by inverse distance
knn_clf_params = {
    'n_neighbors': [3, 5, 10],
    'weights': ['uniform', 'distance']
}

#Initialize GridSearchCV with 5-fold cross-validation and F1 scoring
grid_knn_clf = GridSearchCV(
    KNeighborsClassifier(),
    knn_clf_params,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

#Fit the model on the training set
grid_knn_clf.fit(X_train_clf_scaled, y_train_clf)

# Retrieve the best KNN model from the search
knn_clf_best = grid_knn_clf.best_estimator_

#Make predictions on training, validation, and test sets
knn_clf_train_pred = knn_clf_best.predict(X_train_clf_scaled)
knn_clf_val_pred = knn_clf_best.predict(X_val_clf_scaled)
knn_clf_test_pred = knn_clf_best.predict(X_test_clf_scaled)

#Evaluate performance using your custom metric function
knn_clf_results = print_clf_metrics(
    "KNN (Grid Search)",
    y_train_clf, knn_clf_train_pred,
    y_val_clf, knn_clf_val_pred,
    y_test_clf, knn_clf_test_pred
)

#Print the best hyperparameters found
print("Best KNN Params:", grid_knn_clf.best_params_)

KNN (Grid Search) Classification:
Train Acc: 0.9768, Val Acc: 0.9459, Test Acc: 0.9492
Train F1: 0.9834, Val F1: 0.9614, Test F1: 0.9641

Best KNN Params: {'n_neighbors': 5, 'weights': 'uniform'}


## Classification Results

In [ ]:
#Combine all classification results into a list for easy comparison or tabulation
clf_results = [lr_clf_results, rf_clf_results, xgb_clf_results, nn_clf_results, knn_clf_results]

# Performance Table

In [ ]:
#Convert the list of regression results into a DataFrame
reg_df = pd.DataFrame(reg_results)

#Set the model name as the index for a cleaner table layout
reg_df.set_index("Model", inplace=True)

#Display the regression performance table with color gradient for better visual representation
print("Regression Performance Table")
display(reg_df.style.background_gradient(cmap='YlGnBu', axis=0).format("{:.4f}"))

#Convert the list of classification results into a DataFrame
clf_df = pd.DataFrame(clf_results)

#Set the model name as the index for a cleaner table layout
clf_df.set_index("Model", inplace=True)

#Display the classification performance table with color gradient for better visual representation
print("Classification Performance Table (Healthy)")
display(clf_df.style.background_gradient(cmap='YlGnBu', axis=0).format("{:.4f}"))

Regression Performance Table


,Train RMSE,Val RMSE,Test RMSE,Train R2,Val R2,Test R2
Model,,,,,,
Linear Regression,5466362.6472,4757050.5425,5742351.2068,0.6035,0.7345,0.5355
Random Forest (Grid Search),330531.9558,919789.5327,650026.3435,0.9986,0.9901,0.9940
XGBoost (Random Search),150218.1645,338142.2317,313629.5547,0.9997,0.9987,0.9986
Neural Network (Grid Search),621221.7761,573281.2605,615925.4197,0.9949,0.9961,0.9947
KNN (Bayesian Opt),0.0000,399374.3619,621618.5679,1.0000,0.9981,0.9946


Classification Performance Table (Healthy)


,Train Acc,Val Acc,Test Acc,Train F1,Val F1,Test F1
Model,,,,,,
Logistic Regression (Grid Search),0.8189,0.8226,0.8238,0.8740,0.8763,0.8780
Random Forest (Random Search),1.0000,0.9996,0.9994,1.0000,0.9997,0.9995
XGBoost (Bayesian Opt),1.0000,0.9991,0.9991,1.0000,0.9994,0.9994
Neural Network,0.8007,0.7995,0.7994,0.8403,0.8386,0.8401
KNN (Grid Search),0.9768,0.9459,0.9492,0.9834,0.9614,0.9641
